# 01 — Preprocessing

Loads raw UCSF-PDGM T2_bias NIfTI files, applies z-score normalization on brain voxels only,
resizes to 96³ using proper 3D interpolation, and saves per-patient `.npy` files.
Builds a manifest CSV with type + grade labels and a stratified 70/15/15 train/val/test split.

**Input:** `T2biasCollected/` — raw, untouched UCSF-PDGM T2_bias NIfTIs (already skull-stripped + bias-corrected by UCSF)

**Output:** `preprocessed/` folder with `.npy` files + `manifest.csv`

**Run before:** `02_eda.ipynb`

---
**DO NOT** run skull stripping or N4 bias correction — UCSF already did both. The `_bias` suffix confirms it.

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install nibabel scipy tqdm -q

In [ ]:
# ── Cell 2: Imports & Drive mount ─────────────────────────────────────────────
import os, re, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
from scipy.ndimage import zoom
from sklearn.model_selection import StratifiedShuffleSplit
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
# ── Cell 3: ✏️  Config ────────────────────────────────────────────────────────

# Raw T2_bias NIfTIs — confirmed original UCSF-PDGM data, never modified
RAW_DIR  = '/content/drive/MyDrive/GlioGradev2/T2biasCollected-20260521T040640Z-3-001/T2biasCollected'

# UCSF-PDGM metadata CSV with WHO 2021 labels
META_CSV = '/content/drive/MyDrive/GlioGradev2/UCSF-PDGM-metadata_v2_Fixed.csv'

# Where preprocessed .npy files will be saved
OUT_DIR  = '/content/drive/MyDrive/GlioGradev2/preprocessed'

# Target volume shape for the CNN input
TARGET_SHAPE = (96, 96, 96)

SEED = 42

# WHO 2021 type label mapping  (must match metadata CSV exactly)
TYPE_LABELS = {
    'Glioblastoma, IDH-wildtype':                        0,
    'Astrocytoma, IDH-wildtype':                         1,
    'Oligodendroglioma, IDH-mutant, 1p/19q-codeleted':  2,
    'Astrocytoma, IDH-mutant':                           3,
}

# Grade label mapping
GRADE_LABELS = {2: 0, 3: 1, 4: 2}

print('Config ready.')
print(f'  RAW_DIR  : {RAW_DIR}')
print(f'  META_CSV : {META_CSV}')
print(f'  OUT_DIR  : {OUT_DIR}')
print(f'  Target   : {TARGET_SHAPE}')

In [ ]:
# ── Cell 4: Load metadata & validate labels ────────────────────────────────────
meta = pd.read_csv(META_CSV)
print(f'Metadata shape: {meta.shape}')
print(f'Columns: {list(meta.columns)}\n')

# Normalise ID column to bare integer string '0004', '0005', ...
meta['id_clean'] = meta['ID'].str.replace('UCSF-PDGM-', '', regex=False).str.zfill(4)

DIAG_COL  = 'Final pathologic diagnosis (WHO 2021)'
GRADE_COL = 'WHO CNS Grade'

# Keep only the 4 known WHO 2021 classes
before = len(meta)
meta = meta[meta[DIAG_COL].isin(TYPE_LABELS)].copy()
dropped = before - len(meta)
print(f'Dropped {dropped} rows with misc/unknown labels. {len(meta)} patients remain.\n')

# Drop rows where grade is missing or not in {2, 3, 4}
meta = meta[meta[GRADE_COL].isin(GRADE_LABELS)].copy()
print(f'After grade filter: {len(meta)} patients.\n')

meta['type_idx']  = meta[DIAG_COL].map(TYPE_LABELS)
meta['grade_idx'] = meta[GRADE_COL].map(GRADE_LABELS)

# Class distribution
print('── Type distribution ──────────────────────')
for label, idx in TYPE_LABELS.items():
    n = (meta['type_idx'] == idx).sum()
    bar = '█' * (n // 5)
    print(f'  [{idx}] {label[:50]:<50}  {n:>4}  ({100*n/len(meta):.1f}%)  {bar}')

print('\n── Grade distribution ─────────────────────')
for grade, idx in GRADE_LABELS.items():
    n = (meta['grade_idx'] == idx).sum()
    bar = '█' * (n // 5)
    print(f'  Grade {grade}: {n:>4}  ({100*n/len(meta):.1f}%)  {bar}')

majority_type  = meta['type_idx'].value_counts().iloc[0] / len(meta)
majority_grade = meta['grade_idx'].value_counts().iloc[0] / len(meta)
print(f'\n⚠️  Majority-class baseline — Type: {majority_type:.1%}  Grade: {majority_grade:.1%}')
print('   (A model predicting only the majority class would score this on plain accuracy.)')

In [ ]:
# ── Cell 5: Scan RAW_DIR & cross-reference with metadata ──────────────────────
raw_dir = Path(RAW_DIR)
if not raw_dir.exists():
    raise FileNotFoundError(f'RAW_DIR not found: {RAW_DIR}\nCheck the path in Cell 3.')

# Find all T2_bias files
nii_files = sorted(raw_dir.glob('*_T2_bias.nii.gz'))
print(f'NIfTI files found in RAW_DIR: {len(nii_files)}')

# Extract 4-digit IDs from filenames
def extract_id(fname):
    m = re.search(r'UCSF-PDGM-(\d{4})', fname)
    return m.group(1) if m else None

scan_ids = {extract_id(f.name): f for f in nii_files if extract_id(f.name)}
meta_ids = set(meta['id_clean'])

usable     = meta_ids & set(scan_ids)
scan_only  = set(scan_ids) - meta_ids
meta_only  = meta_ids - set(scan_ids)

print(f'\nCross-reference:')
print(f'  ✅  Have both scan + label (usable): {len(usable)}')
print(f'  ⚠️   Scan file but no label:          {len(scan_only)}')
print(f'  ⚠️   Label but no scan file:           {len(meta_only)}')

# Filter metadata to usable patients only
meta_usable = meta[meta['id_clean'].isin(usable)].copy().reset_index(drop=True)
meta_usable['nii_path'] = meta_usable['id_clean'].map(lambda i: str(scan_ids[i]))
print(f'\nProceeding with {len(meta_usable)} patients.')

In [ ]:
# ── Cell 6: Preprocessing function ────────────────────────────────────────────

def preprocess_volume(nii_path, target_shape=TARGET_SHAPE):
    """
    Load a T2_bias NIfTI, apply z-score norm on brain voxels,
    clip to [-3, 3], rescale to [0, 1], resize to target_shape.
    Returns float32 numpy array of shape target_shape.
    """
    img  = nib.load(str(nii_path))
    vol  = img.get_fdata(dtype=np.float32)

    # Z-score normalisation on brain voxels only (skull voxels are 0)
    brain_mask = vol > 0
    if brain_mask.sum() == 0:
        raise ValueError('No non-zero voxels — file may be empty or corrupt.')

    mean = vol[brain_mask].mean()
    std  = vol[brain_mask].std()
    vol  = (vol - mean) / (std + 1e-8)

    # Clip outliers, rescale to [0, 1]
    vol = np.clip(vol, -3.0, 3.0)
    vol = (vol + 3.0) / 6.0

    # Resize to target_shape using 3D zoom (order=1 = trilinear)
    factors = tuple(t / s for t, s in zip(target_shape, vol.shape))
    vol = zoom(vol, factors, order=1).astype(np.float32)

    # Sanity check
    assert vol.shape == target_shape, f'Shape mismatch: {vol.shape} != {target_shape}'
    assert 0.0 <= vol.min() and vol.max() <= 1.0, 'Value range out of [0, 1]'

    return vol


# Quick smoke test on the first file
test_path = meta_usable['nii_path'].iloc[0]
test_vol  = preprocess_volume(test_path)
print(f'Smoke test passed:')
print(f'  Input file : {Path(test_path).name}')
print(f'  Output shape: {test_vol.shape}')
print(f'  Value range : [{test_vol.min():.4f}, {test_vol.max():.4f}]')
print(f'  Zero fraction: {(test_vol == 0).mean():.1%}')
print(f'  dtype: {test_vol.dtype}')

In [ ]:
# ── Cell 7: Run preprocessing ──────────────────────────────────────────────────
# Saves one .npy file per patient to OUT_DIR.
# Skips patients that are already saved (safe to re-run).

out_dir = Path(OUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

npy_paths = []
failures  = []

for _, row in tqdm(meta_usable.iterrows(), total=len(meta_usable), desc='Preprocessing'):
    pid      = row['id_clean']
    npy_path = out_dir / f'UCSF-PDGM-{pid}.npy'

    if npy_path.exists():
        npy_paths.append(str(npy_path))
        continue

    try:
        vol = preprocess_volume(row['nii_path'])
        np.save(str(npy_path), vol)
        npy_paths.append(str(npy_path))
    except Exception as e:
        print(f'  ❌  UCSF-PDGM-{pid}: {e}')
        failures.append({'id': pid, 'error': str(e)})
        npy_paths.append(None)

meta_usable['npy_path'] = npy_paths

# Drop failed patients
meta_ok = meta_usable[meta_usable['npy_path'].notna()].copy().reset_index(drop=True)

print(f'\n✅  Saved: {len(meta_ok)} / {len(meta_usable)} patients')
if failures:
    print(f'❌  Failed: {len(failures)}')
    for f in failures:
        print(f'    {f["id"]}: {f["error"]}')

In [ ]:
# ── Cell 8: Stratified 70/15/15 split + manifest CSV ─────────────────────────
# Split is stratified on type_idx (4-class) — more granular than grade,
# so grade distribution is also approximately preserved.

# First split: carve off 30% as temp (→ val + test)
sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
train_idx, temp_idx = next(sss1.split(meta_ok, meta_ok['type_idx']))

meta_train = meta_ok.iloc[train_idx].copy()
meta_temp  = meta_ok.iloc[temp_idx].copy()

# Second split: split temp 50/50 → val / test
sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
val_idx, test_idx = next(sss2.split(meta_temp, meta_temp['type_idx']))

meta_val  = meta_temp.iloc[val_idx].copy()
meta_test = meta_temp.iloc[test_idx].copy()

meta_train['split'] = 'train'
meta_val['split']   = 'val'
meta_test['split']  = 'test'

manifest = pd.concat([meta_train, meta_val, meta_test], ignore_index=True)

# Select and rename columns for the manifest
manifest = manifest[[
    'id_clean', 'npy_path', 'split',
    DIAG_COL, 'type_idx',
    GRADE_COL, 'grade_idx'
]].rename(columns={
    'id_clean': 'patient_id',
    DIAG_COL:  'type_label',
    GRADE_COL: 'grade_label',
})

manifest_path = out_dir / 'manifest.csv'
manifest.to_csv(str(manifest_path), index=False)

print(f'Manifest saved → {manifest_path}')
print(f'\nSplit sizes:')
print(f'  Train : {len(meta_train):>4}  ({100*len(meta_train)/len(meta_ok):.1f}%)')
print(f'  Val   : {len(meta_val):>4}  ({100*len(meta_val)/len(meta_ok):.1f}%)')
print(f'  Test  : {len(meta_test):>4}  ({100*len(meta_test)/len(meta_ok):.1f}%)')

print(f'\nType distribution per split:')
for split in ['train', 'val', 'test']:
    subset = manifest[manifest['split'] == split]
    counts = subset['type_label'].value_counts()
    print(f'  {split}:')
    for label, n in counts.items():
        print(f'    {label[:45]:<45} {n:>3}')

In [ ]:
# ── Cell 9: Verify outputs ─────────────────────────────────────────────────────
# Loads 3 random .npy files and checks shape, range, and plots a middle slice.

sample = manifest.sample(3, random_state=SEED)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
fig.suptitle('Verification — middle axial slice of 3 random preprocessed volumes', fontsize=11)

for ax, (_, row) in zip(axes, sample.iterrows()):
    vol = np.load(row['npy_path'])
    mid = vol.shape[2] // 2

    # Checks
    assert vol.shape == TARGET_SHAPE, f'Shape {vol.shape} != {TARGET_SHAPE}'
    assert vol.dtype == np.float32,   f'dtype {vol.dtype} != float32'
    assert vol.min() >= 0.0 and vol.max() <= 1.0, f'Range [{vol.min():.3f}, {vol.max():.3f}]'

    ax.imshow(np.rot90(vol[:, :, mid]), cmap='gray', vmin=0, vmax=1)
    ax.set_title(
        f'ID: {row["patient_id"]}\n'
        f'{row["type_label"].split(",")[0]}\n'
        f'Grade {row["grade_label"]}  |  split: {row["split"]}',
        fontsize=7
    )
    ax.axis('off')

    print(f'  {row["patient_id"]}: shape={vol.shape}  range=[{vol.min():.3f}, {vol.max():.3f}]'
          f'  zeros={( vol==0).mean():.1%}  dtype={vol.dtype}')

plt.tight_layout()
plt.show()
print('\n✅  All verification checks passed.')

In [ ]:
# ── Cell 10: Summary ───────────────────────────────────────────────────────────
SEP = '═' * 60
print(SEP)
print('  PREPROCESSING COMPLETE')
print(SEP)
print(f'  Patients processed : {len(meta_ok)}')
print(f'  Failures           : {len(failures)}')
print(f'  Output shape       : {TARGET_SHAPE}  (float32)')
print(f'  Normalization      : z-score on brain voxels → clip [-3,3] → [0,1]')
print(f'  Resize method      : scipy.ndimage.zoom  order=1 (trilinear)')
print(f'  Split              : 70 / 15 / 15  (stratified on 4-class type label)')
print(f'  Manifest CSV       : {manifest_path}')
print()
print('  Grade distribution across splits:')
print(manifest.groupby(['split', 'grade_label']).size().unstack(fill_value=0).to_string())
print()
print('  Steps skipped (done by UCSF):')
print('    ✅  Skull stripping')
print('    ✅  N4 bias field correction')
print('    ✅  Co-registration to 1mm isotropic space')
print()
if failures:
    print(f'  ⚠️   Failed patients (excluded from manifest):')
    for f in failures:
        print(f'    {f["id"]}: {f["error"]}')
    print()
print(f'  ➡️   Next: open 02_eda.ipynb')
print(SEP)